# Genome-wide read-depth tracks (primary reads)

Per-chromosome read-depth tracks of HiFi (long) and Illumina (short) reads
mapped as primary alignments to the unmasked and hard-masked degu assembly,
binned to 150 kb. Produces both the raw and rolling-smoothed coverage figures.


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
from natsort import natsorted
import matplotlib.gridspec as gridspec


In [ ]:
input_dir = f"{PROJ_ROOT}/code/command-line-script/contig-coverage/2-primaryreads-coverage/"


In [ ]:
# Long-read coverage, unmasked — primary reads, 15 kb windows.
longcov_15kb = pd.read_csv(
    input_dir + "coverage_long_read_hifiasm_041425_scaffolded_juiceBox_sorted_chrAssigned_primaryReads_15kb_windows.tsv",
    sep="\t", names=["contig", "start_pos", "end_pos", "coverage"])
longcov_15kb = longcov_15kb[longcov_15kb["contig"].str.contains("chr")]

# Bin to 150 kb and average coverage per bin.
bin_size = 150000
longcov_15kb["bin"] = longcov_15kb["start_pos"] // bin_size
longcov = longcov_15kb.groupby(["contig", "bin"]).agg({
    "start_pos": "min", "end_pos": "max", "coverage": "mean"
}).reset_index()[["contig", "start_pos", "end_pos", "coverage"]]


In [ ]:
# Short-read coverage, unmasked — primary reads, 15 kb windows.
shortcov_15kb = pd.read_csv(
    input_dir + "coverage_short_read_hifiasm_041425_scaffolded_juiceBox_sorted_chrAssigned_primaryRead_15kb_windows.tsv",
    sep="\t", names=["contig", "start_pos", "end_pos", "coverage"])
shortcov_15kb = shortcov_15kb[shortcov_15kb["contig"].str.contains("chr")]

# Bin to 150 kb and average coverage per bin.
bin_size = 150000
shortcov_15kb["bin"] = shortcov_15kb["start_pos"] // bin_size
shortcov = shortcov_15kb.groupby(["contig", "bin"]).agg({
    "start_pos": "min", "end_pos": "max", "coverage": "mean"
}).reset_index()[["contig", "start_pos", "end_pos", "coverage"]]


In [ ]:
# Short-read coverage, hard-masked — primary reads, 15 kb windows.
masked_shortcov_15kb = pd.read_csv(
    input_dir + "coverage_short_read_hifiasm_041425_scaffolded_juiceBox_sorted_hardMasked_chrAssigned_primaryRead_15kb_windows.tsv",
    sep="\t", names=["contig", "start_pos", "end_pos", "coverage"])
masked_shortcov_15kb = masked_shortcov_15kb[masked_shortcov_15kb["contig"].str.contains("chr")]

# Bin to 150 kb and average coverage per bin.
bin_size = 150000
masked_shortcov_15kb["bin"] = masked_shortcov_15kb["start_pos"] // bin_size
masked_shortcov = masked_shortcov_15kb.groupby(["contig", "bin"]).agg({
    "start_pos": "min", "end_pos": "max", "coverage": "mean"
}).reset_index()[["contig", "start_pos", "end_pos", "coverage"]]


In [ ]:
# Long-read coverage, hard-masked — primary reads, 15 kb windows.
masked_longcov_15kb = pd.read_csv(
    input_dir + "coverage_long_read_hifiasm_041425_scaffolded_juiceBox_sorted_hardMasked_chrAssigned_primaryRead_15kb_windows_fromCalculateScript.tsv",
    sep="\t", names=["contig", "start_pos", "end_pos", "coverage"])
masked_longcov_15kb = masked_longcov_15kb[masked_longcov_15kb["contig"].str.contains("chr")]

# Bin to 150 kb and average coverage per bin.
bin_size = 150000
masked_longcov_15kb["bin"] = masked_longcov_15kb["start_pos"] // bin_size
masked_longcov = masked_longcov_15kb.groupby(["contig", "bin"]).agg({
    "start_pos": "min", "end_pos": "max", "coverage": "mean"
}).reset_index()[["contig", "start_pos", "end_pos", "coverage"]]


In [ ]:
# Prepare data - assuming these DataFrames exist:
# longcov, masked_longcov, shortcov, masked_shortcov

# Calculate chromosome lengths
all_lengths = pd.concat([
    df.groupby('contig')['end_pos'].max() 
    for df in [longcov, masked_longcov, shortcov, masked_shortcov]
])
chr_lengths = all_lengths.groupby(level=0).max().sort_values(ascending=False)
contigs = natsorted(chr_lengths.index.tolist())
chr_lengths = chr_lengths[contigs]
n_contigs = len(contigs)

# Normalize widths
max_length = chr_lengths.max()
relative_widths = chr_lengths / max_length

# Define datasets
datasets = [
    {'df': longcov, 'color': 'blue', 'name': 'Long read\ncoverage\nUnmasked', 'type': 'Long'},
    {'df': masked_longcov, 'color': 'dodgerblue', 'name': 'Long read\ncoverage\nMasked', 'type': 'Long'}, 
    {'df': shortcov, 'color': 'red', 'name': 'Short read\ncoverage\nUnmasked', 'type': 'Short'},
    {'df': masked_shortcov, 'color': 'tomato', 'name': 'Short read\ncoverage\nMasked', 'type': 'Short'}
]

# Calculate median coverage for each read type
long_median = pd.concat([longcov['coverage'], masked_longcov['coverage']]).median()
short_median = pd.concat([shortcov['coverage'], masked_shortcov['coverage']]).median()

# Set y-limits based on read type
for dataset in datasets:
    dataset['ymin'] = 0  # Coverage can't be negative
    
    if dataset['type'] == 'Long':
        dataset['ymax'] = long_median * 3
    else:  # Short reads
        dataset['ymax'] = short_median * 3
    
# Layout parameters
contigs_per_row = 30  # Reduced for larger plots
n_rows = math.ceil(n_contigs / contigs_per_row) * len(datasets)

# Create figure
fig = plt.figure(figsize=(20, n_rows * 1.5))
gs = gridspec.GridSpec(n_rows, contigs_per_row,
                      width_ratios=relative_widths[:contigs_per_row],
                      hspace=0.1, wspace=0.1)

# Plotting loop
plot_counter = 0
for contig_idx, contig in enumerate(contigs):
    for dataset_idx, dataset in enumerate(datasets):
        row = (contig_idx // contigs_per_row) * len(datasets) + dataset_idx
        col = contig_idx % contigs_per_row
        
        ax = plt.subplot(gs[row, col])
        plot_counter += 1
        
        # Plot data
        contig_data = dataset['df'][dataset['df']['contig'] == contig]
        midpoints = (contig_data['start_pos'] + contig_data['end_pos']) / 2
        ax.plot(midpoints, contig_data['coverage'], color=dataset['color'], linewidth=0.8)
        
        # Set axes limits
        ax.set_ylim(dataset['ymin'], dataset['ymax'])
        ax.set_xlim(0, chr_lengths[contig])
        
        # Configure ticks
        max_mb = chr_lengths[contig]/1e6
        major_ticks = np.arange(0, max_mb+ 1e-9, 50) * 1e6
        minor_ticks = np.arange(0, max_mb+ 1e-9, 10) * 1e6
        
        ax.set_xticks(major_ticks)
        ax.set_xticks(minor_ticks, minor=True)
        ax.set_xlim(0, chr_lengths[contig])
        
        # Smart formatting

        is_first_row_in_group = (row % len(datasets)) == 0
        is_last_contig_group = (contig_idx // contigs_per_row) == (math.ceil(n_contigs / contigs_per_row)) - 1
        is_first_col = col == 0
        
        # Title only on first row of each contig group
        if is_first_row_in_group:
            ax.set_title(contig, fontsize=10, rotation=45, pad=3)
        
        # X-axis labels only on bottom row
        if is_last_contig_group and (dataset_idx == len(datasets)-1):
            # ax.set_xticklabels([f"{x/1e6:.0f}" if x < chr_lengths[contig] else "" for x in major_ticks], fontsize=6)
            # ax.set_xticklabels([f"{x/1e6:.0f}" if x in major_ticks else "" for x in major_ticks], fontsize=6)
            ax.set_xticklabels([f"{x/1e6:.0f}" if (x in major_ticks and x != 0) else "" for x in major_ticks], fontsize=9)

            # ax.tick_params(axis='x', which='both', bottom=True, labelbottom=False)
            ax.set_xlabel(f"{chr_lengths[contig]/1e6:.1f}Mb", fontsize=9, rotation=45)
        else:
            # ax.set_xticklabels([])
            # ax.set_xticks(minor_ticks, minor=True)
            ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
            # ax.set_xlim(0, chr_lengths[contig])
        
        # Y-axis labels only on first column
        if is_first_col:
            ax.set_ylabel(dataset['name'], fontsize=12)
            ax.tick_params(axis='y', labelsize=10)
        else:
            ax.tick_params(axis='y', which='both', left=False, labelleft=False)
        
        # Grid lines
        ax.grid(True, which='major', alpha=0.2)
        ax.grid(True, which='minor', alpha=0.1)

# Hide unused axes
for i in range(plot_counter, n_rows * contigs_per_row):
    row = i // contigs_per_row
    col = i % contigs_per_row
    fig.delaxes(gs[row, col])

plt.subplots_adjust(left=0.06, right=0.98, bottom=0.05, top=0.95)
plt.savefig('coverage_genome_primaryReadOnly.png', dpi=600,bbox_inches="tight")
plt.show()

In [ ]:
# Define your window size (in bases)
window_size = 150000  # Match your bin size for natural smoothing
min_periods = 1      # Minimum points needed for calculation

for dataset in datasets:
    # Sort by position first (critical for rolling)
    dataset['df'] = dataset['df'].sort_values(['contig', 'start_pos'])
    
    # Calculate rolling median coverage per chromosome
    dataset['df']['smoothed_coverage'] = (dataset['df']
        .groupby('contig')['coverage']
        .transform(lambda x: x.rolling(window=window_size//15000,  # Number of bins
                                 min_periods=min_periods,
                                 center=True).mean())
    )
    
    # Fill NA values at chromosome ends
    dataset['df']['smoothed_coverage'] = dataset['df']['smoothed_coverage'].fillna(
        dataset['df']['coverage']
    )

In [ ]:
## Plotting smoothed coverage
# Calculate chromosome lengths
all_lengths = pd.concat([
    df.groupby('contig')['end_pos'].max() 
    for df in [longcov, masked_longcov, shortcov, masked_shortcov]
])
chr_lengths = all_lengths.groupby(level=0).max().sort_values(ascending=False)
contigs = natsorted(chr_lengths.index.tolist())
chr_lengths = chr_lengths[contigs]
n_contigs = len(contigs)

# Normalize widths
max_length = chr_lengths.max()
relative_widths = chr_lengths / max_length


# Calculate median coverage for each read type
long_median = pd.concat([longcov['coverage'], masked_longcov['coverage']]).median()
short_median = pd.concat([shortcov['coverage'], masked_shortcov['coverage']]).median()

# Set y-limits based on read type
for dataset in datasets:
    dataset['ymin'] = 0  # Coverage can't be negative
    
    if dataset['type'] == 'Long':
        dataset['ymax'] = long_median * 3
    else:  # Short reads
        dataset['ymax'] = short_median * 3
    
# Layout parameters
contigs_per_row = 30  # Reduced for larger plots
n_rows = math.ceil(n_contigs / contigs_per_row) * len(datasets)

# Create figure
fig = plt.figure(figsize=(20, n_rows * 1.2))
gs = gridspec.GridSpec(n_rows, contigs_per_row,
                      width_ratios=relative_widths[:contigs_per_row],
                      hspace=0.1, wspace=0.1)

# Plotting loop
plot_counter = 0
for contig_idx, contig in enumerate(contigs):
    for dataset_idx, dataset in enumerate(datasets):
        row = (contig_idx // contigs_per_row) * len(datasets) + dataset_idx
        col = contig_idx % contigs_per_row
        
        ax = plt.subplot(gs[row, col])
        plot_counter += 1
        
        # Plot data
        contig_data = dataset['df'][dataset['df']['contig'] == contig]
        midpoints = (contig_data['start_pos'] + contig_data['end_pos']) / 2
        # With the smoothed version:
        ax.plot(midpoints, contig_data['smoothed_coverage'], 
                color=dataset['color'], linewidth=0.8) #, alpha=0.8)
        
        # # Keep original as thin line underneath
        # ax.plot(midpoints, contig_data['coverage'], 
        #         color=dataset['color'], linewidth=0.3, alpha=0.3)      
        
        # Set axes limits
        ax.set_ylim(dataset['ymin'], dataset['ymax'])
        
        # Configure ticks
        max_mb = chr_lengths[contig]/1e6
        major_ticks = np.arange(0, max_mb+ 1e-9, 50) * 1e6
        minor_ticks = np.arange(0, max_mb+ 1e-9, 10) * 1e6
        
        ax.set_xticks(major_ticks)
        ax.set_xticks(minor_ticks, minor=True)
        ax.set_xlim(0, chr_lengths[contig])
        
        # Smart formatting

        is_first_row_in_group = (row % len(datasets)) == 0
        is_last_contig_group = (contig_idx // contigs_per_row) == (math.ceil(n_contigs / contigs_per_row)) - 1
        is_first_col = col == 0
        
        # Title only on first row of each contig group
        if is_first_row_in_group:
            ax.set_title(contig, fontsize=10, rotation=45, pad=3)
        
        # X-axis labels only on bottom row
        if is_last_contig_group and (dataset_idx == len(datasets)-1):
            ax.set_xticklabels([f"{x/1e6:.0f}" if (x in major_ticks and x != 0) else "" for x in major_ticks], fontsize=9)
            ax.set_xlabel(f"{chr_lengths[contig]/1e6:.1f}Mb", fontsize=9, rotation=45)
        else:
            # ax.set_xticklabels([])
            # ax.set_xticks(minor_ticks, minor=True)
            ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
            # ax.set_xlim(0, chr_lengths[contig])
        
        # Y-axis labels only on first column
        if is_first_col:
            ax.set_ylabel(dataset['name'], fontsize=12)
            ax.tick_params(axis='y', labelsize=10)
        else:
            ax.tick_params(axis='y', which='both', left=False, labelleft=False)
        
        # Grid lines
        ax.grid(True, which='major', alpha=0.2)
        ax.grid(True, which='minor', alpha=0.1)

# Hide unused axes
for i in range(plot_counter, n_rows * contigs_per_row):
    row = i // contigs_per_row
    col = i % contigs_per_row
    fig.delaxes(gs[row, col])

plt.subplots_adjust(left=0.06, right=0.98, bottom=0.01, top=0.95,hspace=0.01)
plt.savefig('coverage_genome_rolling_primaryRead.png', dpi=600,bbox_inches="tight") 
plt.show()